# ⚡ Losses

This notebook shows how to use the various losses defined in this project.

## Setup

---

Let's install some necessary dependencies and set global variables.

In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
# Autoroot
import autorootcwd

In [3]:
# Imports
import torch
from hydra import compose, initialize
from hydra.utils import instantiate

/home/senghaas/.conda/envs/sillystill/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
x = torch.randn(1, 3, 128, 128)
y = torch.randn(1, 3, 128, 128)

## Individual Losses

In [5]:
from src.models.loss import MAELoss, MSELoss, VGGLoss, ColorLoss, GCLMLoss, FrequencyLoss, TVAbsoluteLoss, TVRelativeLoss, CoBiLoss

### MAE

The `MAELoss` is the mean absolute error between two tensors.

In [6]:
# Initialise `MAELoss`
mae = MAELoss()
loss1, loss2 = mae(x, x), mae(x, y)
assert loss1 < loss2

loss1, loss2

(tensor(0.), tensor(1.1299))

### MSE

The `MSELoss` is the mean squared error between two tensors.

In [7]:
# Initialise `MSELoss`
mse = MSELoss()
loss1, loss2 = mse(x, x), mse(x, y)
assert loss1 < loss2

loss1, loss2

(tensor(0.), tensor(2.0069))

### VGG

In [8]:
# Initialise `VGGLoss`
vgg = VGGLoss()
loss1, loss2 = vgg(x, x), vgg(x, y)
assert loss1 < loss2

loss1, loss2

(tensor(0., grad_fn=<AddBackward0>), tensor(0.1823, grad_fn=<AddBackward0>))

### ColorLoss

In [9]:
# Initialise `ColorLoss`
color = ColorLoss()
loss1, loss2 = color(x, x), color(x, y)
loss3 = color(x, torch.ones_like(x)) # noise to white
assert loss1 < loss2
assert loss1 < loss3

loss1, loss2, loss3

(tensor(0.), tensor(0.0452), tensor(1.0198))

### GCLMLoss

In [10]:
# Initialise `GCLMLoss`
gclm = GCLMLoss()
loss1, loss2 = gclm(x, x), gclm(x, y)
assert loss1 < loss2

loss1, loss2

(tensor(0., dtype=torch.float64), tensor(397.4519, dtype=torch.float64))

### Frequency Loss

In [11]:
# Initialise `Frequency Loss`
freq = FrequencyLoss()
loss1, loss2 = freq(x, x), freq(x, y)
assert loss1 < loss2

loss1, loss2

(tensor(0.), tensor(7028.2837))

### TVAbsolute Loss

In [12]:
# Initialise `TVAbsoluteLoss`
abs_tv = TVAbsoluteLoss()
loss1 = abs_tv(x, ...) # noisy image
loss2 = abs_tv(torch.ones(1, 3, 128, 128), ...) # uniform image
assert loss1 < loss2

loss1, loss2

(tensor(0.7763), tensor(100000.))

### TVRelative Loss

In [13]:
# Initialise `TVRelativeLoss`
rel_tv = TVRelativeLoss()
loss1, loss2 = rel_tv(x, x), rel_tv(x, y)
assert loss1 < loss2

loss1, loss2

(tensor(0.), tensor(0.0005))

### CoBi Loss

In [14]:
# Initialise `CoBiLoss`
cobi = CoBiLoss()
loss1, loss2 = cobi(x, x), cobi(x, y)
assert loss1 < loss2

loss1, loss2

(tensor(0.0123), tensor(2.2459))

### AutoTranslateLoss

In [15]:
# Initialise `AutoTranslateLoss`
# auto_translate = AutoTranslateLoss()
# TODO: Show how to use

## Combined Losses (SillyLoss)

In [22]:
# Initialise `SillyLoss`
from src.models.loss import SillyLoss

silly = SillyLoss(losses=[mse, mae], weights=[1.0, 1.0])
loss1, loss2 = silly(x, x), silly(x, y)

assert loss1["loss"] < loss2["loss"]
loss1, loss2

({'mse_loss': tensor(0.), 'mae_loss': tensor(0.), 'loss': tensor(0.)},
 {'mse_loss': tensor(2.0069),
  'mae_loss': tensor(1.1299),
  'loss': tensor(3.1368)})

## Hydra

In [17]:
# TODO: Hydra doesn't work yet
for loss in ["mse", "cobi", "simple-combined", "auto-translate"]:
    with initialize(version_base=None, config_path="../configs/model/loss", job_name="losses"):
        cfg = compose(config_name=loss)
        loss = instantiate(cfg)
        print(f"✅ Initialised {cfg._target_}")

✅ Initialised src.models.loss.MSELoss
✅ Initialised src.models.loss.CoBiLoss


MissingConfigException: Cannot find primary config 'simple-combined'. Check that it's in your config search path.

Config search path:
	provider=hydra, path=pkg://hydra.conf
	provider=main, path=file:///scratch/senghaas/sillystill/configs/model/loss
	provider=hydra-colorlog, path=pkg://hydra_plugins.hydra_colorlog.conf
	provider=schema, path=structured://